In [30]:
from google.colab import drive
import os
import torch
from torch.utils.data import Dataset, DataLoader
drive.mount('/content/drive')
import cv2
from google.colab.patches import cv2_imshow
from torch.utils.data import DataLoader
import numpy as np
import torchvision.transforms as transforms
import timm

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [31]:
ROOT = "/content/drive/MyDrive/Digital_Knee_X_ray_Images"
TRAIN_DATASET_DIR = "MedicalExpert-I"
VALIDATE_DATASET_DIR = "MedicalExpert-II"

In [32]:
class KneeXRayDataset(Dataset):

    def load_images_path(self):
        #iter each category
        # print(self.categories)
        for i, category in enumerate(self.categories):
            category_path = os.path.join(self.root, category)

            #iter file in category
            for file_path in os.listdir(category_path):
                self.image_paths.append(os.path.join(category_path, file_path))
                self.labels.append(i)

    def load_image_from_path(self,imagePath):
        return cv2.imread(imagePath)
        
    def __init__(self, root, train_dataset_dir, validate_dataset_dir, transform , train=True, ):
        self.root = root
        self.transform  = transform

        # determine which dir will use (train , val)
        if train :
            self.root = os.path.join(root, train_dataset_dir)
        else:
            self.root = os.path.join(root,validate_dataset_dir)

        # get all category (0 -> 4)
        self.categories = os.listdir(self.root)
        self.image_paths = []
        self.labels = []

        # load images from dataset
        self.load_images_path()

        
        
    def __len__(self):
        return len(self.labels)
        
    def __getitem__(self, idx):
        image = self.load_image_from_path(self.image_paths[idx])
        # cv2_imshow(image)
        # cv2.waitKey(0)
        # cv2.destroyAllWindows()
        return self.transform(image), self.labels[idx]



In [33]:
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Resize((224,224))    
])

In [34]:
train_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, transform=transform)
validate_dataset = KneeXRayDataset(root=ROOT, train_dataset_dir=TRAIN_DATASET_DIR, validate_dataset_dir=VALIDATE_DATASET_DIR, train=False , transform=transform)

In [35]:
train_dataset_loader =  DataLoader(train_dataset, batch_size=16, shuffle=True , drop_last=False)
validate_dataset_loader = DataLoader(validate_dataset, batch_size=16, drop_last=False)

In [36]:
images, label = next(iter(train_dataset_loader))
# images = np.transpose(images, (2,0,1))
# cv2_imshow(images.numpy())
# cv2.waitKey(0)
print(images.shape)
print(label)

# for images,labels in train_dataset:
#     print(images.shape)
#     print(labels)

torch.Size([16, 3, 224, 224])
tensor([1, 0, 0, 2, 0, 2, 0, 2, 4, 3, 2, 2, 2, 0, 2, 4])


In [39]:
#define transfer learning model function
def create_transfer_learning_model(model_name: str="efficientnet_b0", pretrained: bool = True, num_classes:int = 5):
    model = timm.create_model(model_name, pretrained= pretrained, num_classes=5)

    #frezze all other layer
    for param in model.parameters():
        param.requires_grad = False

    #unfrezze the class classifier
    for param in model.classifier.parameters():
        param.requires_grad = True

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    return model.to(device)


efficientnet_b0_model = create_transfer_learning_model()

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:122: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


model.safetensors:   0%|          | 0.00/21.4M [00:00<?, ?B/s]

In [41]:
# define activation function & loss function